01. Limpiar y ordenar TiemposRuta - CDG y SFOR

In [1]:
#Librerias y funciones
import os
import pandas as pd
from datetime import datetime

# Función para procesar un archivo de TiemposRuta
def procesar_tiempos_ruta(ruta_carpeta, origen):
    """
    Procesa archivos de TiemposRuta desde una carpeta específica.
    
    Args:
        ruta_carpeta: Ruta de la carpeta donde está el archivo
        origen: Identificador del origen ('CDG' o 'SFOR')
    
    Returns:
        DataFrame procesado o None si no hay archivos
    """
    print(f"\n{'='*60}")
    print(f"PROCESANDO: {origen}")
    print(f"{'='*60}")
    
    # Buscar el primer archivo Excel o CSV en la carpeta
    if not os.path.exists(ruta_carpeta):
        print(f"❌ La carpeta no existe: {ruta_carpeta}")
        return None
    
    archivos = [f for f in os.listdir(ruta_carpeta) if f.endswith(('.xlsx', '.xls', '.csv'))]
    
    if not archivos:
        print(f"❌ No se encontró ningún archivo Excel o CSV en {origen}")
        return None
    
    archivo = archivos[0]
    ruta_archivo = os.path.join(ruta_carpeta, archivo)
    
    print(f"📄 Archivo encontrado: {archivo}")
    
    # Leer archivo
    if archivo.endswith('.csv'):
        tiempos_ruta = pd.read_csv(ruta_archivo, header=6)
    else:
        tiempos_ruta = pd.read_excel(ruta_archivo, header=6)
    
    # Elimina todas las columnas que contienen 'Unnamed:'
    tiempos_ruta = tiempos_ruta.loc[:, ~tiempos_ruta.columns.str.contains('^Unnamed')]
    
    # Separar todas las columnas en _A y _B según el criterio solicitado
    df_sep = pd.DataFrame()
    for col in tiempos_ruta.columns:
        col_A = []
        col_B = []
        for i, val in enumerate(tiempos_ruta[col]):
            if i % 2 == 0:
                col_A.append(val)
            else:
                col_B.append(val)
        # Rellenar para igualar longitudes
        max_len = max(len(col_A), len(col_B))
        col_A += [None] * (max_len - len(col_A))
        col_B += [None] * (max_len - len(col_B))
        df_sep[col + '_A'] = col_A
        df_sep[col + '_B'] = col_B
    
    # Renombrar columnas según lo solicitado
    renombrar = {
        'Tipo Unidad/Partidas_A': 'Tipo de Unidad',
        'Ruta/Nombre Ruta_A': 'Ruta',
        'Ruta/Nombre Ruta_B': 'Nombre Ruta',
        'Hoja Ruta/ Placas_A': 'SAP',
        'Hoja Ruta/ Placas_B': 'Placa',
        'Cierre_A': 'Cierre',
        'Total Escaneos_A': 'Escaneos'
    }
    
    # Seleccionar solo las columnas que se van a conservar y renombrar
    columnas_finales = [col for col in renombrar if col in df_sep.columns]
    df_sep = df_sep[columnas_finales].rename(columns=renombrar)
    
    # Obtener año actual dinámicamente
    anio_actual = str(datetime.now().year)
    
    # Crear columna 'Ruta Conca' concatenando año actual y ruta
    df_sep.insert(
        df_sep.columns.get_loc('Nombre Ruta'),  # Inserta antes de 'Nombre Ruta'
        'Ruta Conca',
        anio_actual + df_sep['Ruta'].astype(str)
    )
    
    # Quitar los primeros 3 ceros de la columna SAP
    if 'SAP' in df_sep.columns:
        df_sep['SAP'] = df_sep['SAP'].astype(str).str.replace('^000', '', regex=True)
    
    # Agregar columna de origen
    df_sep['Origen'] = origen
    
    print(f"✅ Procesado exitosamente")
    print(f"   📊 Filas: {len(df_sep)}")
    print(f"   📋 Columnas: {len(df_sep.columns)}")
    
    return df_sep

# Rutas de las carpetas de entrada
ruta_cdg = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta METSyDIRECTOS\Inputs\CDG"
ruta_sfor = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta METSyDIRECTOS\Inputs\SFOR"

# Procesar ambos archivos
df_cdg = procesar_tiempos_ruta(ruta_cdg, 'CDG')
df_sfor = procesar_tiempos_ruta(ruta_sfor, 'SFOR')

# Consolidar ambos DataFrames
dataframes_procesados = [df for df in [df_cdg, df_sfor] if df is not None]

if dataframes_procesados:
    df_sep = pd.concat(dataframes_procesados, ignore_index=True)
    
    print(f"\n{'='*60}")
    print(f"CONSOLIDACIÓN FINAL")
    print(f"{'='*60}")
    print(f"✅ Total de filas consolidadas: {len(df_sep)}")
    print(f"📊 Distribución por origen:")
    print(df_sep['Origen'].value_counts())
    print(f"\n📋 Columnas finales: {list(df_sep.columns)}")
else:
    print("\n❌ No se pudo procesar ningún archivo")
    df_sep = pd.DataFrame()

# Renombrar df_sep a df_filtrado_sf para compatibilidad con el resto del código
df_filtrado_sf = df_sep.copy()

# 🔧 CONVERTIR COLUMNA CIERRE A DATETIME CON FORMATO CORRECTO (dd/mm/yyyy)
if not df_filtrado_sf.empty and 'Cierre' in df_filtrado_sf.columns:
    print(f"\n🔄 Convirtiendo columna 'Cierre' a datetime con formato dd/mm/yyyy HH:MM:SS...")
    df_filtrado_sf['Cierre'] = pd.to_datetime(
        df_filtrado_sf['Cierre'], 
        format='%d/%m/%Y %H:%M:%S',
        errors='coerce'
    )
    
    # Verificar conversión
    nulos = df_filtrado_sf['Cierre'].isna().sum()
    if nulos > 0:
        print(f"⚠️  {nulos} valores de Cierre no se pudieron convertir (ahora son NaT)")
    else:
        print(f"✅ Todos los valores de Cierre convertidos exitosamente")

# Ordenar por las columnas 'Placa' y 'Cierre' de la A a la Z
if not df_filtrado_sf.empty:
    df_filtrado_sf = df_filtrado_sf.sort_values(by=['Placa', 'Cierre'], ascending=[True, True]).reset_index(drop=True)
    print(f"\n✅ DataFrame ordenado por Placa y Cierre")



PROCESANDO: CDG
📄 Archivo encontrado: Copia de TiemposRutaExcel CDG17mar.xls
✅ Procesado exitosamente
   📊 Filas: 111
   📋 Columnas: 9

PROCESANDO: SFOR
📄 Archivo encontrado: Copia de TiemposRutaExcel SFOR17mar.xls
✅ Procesado exitosamente
   📊 Filas: 757
   📋 Columnas: 9

CONSOLIDACIÓN FINAL
✅ Total de filas consolidadas: 868
📊 Distribución por origen:
Origen
SFOR    757
CDG     111
Name: count, dtype: int64

📋 Columnas finales: ['Tipo de Unidad', 'Ruta', 'Ruta Conca', 'Nombre Ruta', 'SAP', 'Placa', 'Cierre', 'Escaneos', 'Origen']

🔄 Convirtiendo columna 'Cierre' a datetime con formato dd/mm/yyyy HH:MM:SS...
✅ Todos los valores de Cierre convertidos exitosamente

✅ DataFrame ordenado por Placa y Cierre


2. Identificar tipo de cliente (MET o DIRECTO) desde Monitor METS-DIRECTOS

In [2]:
# Identificar tipo de cliente (MET o DIRECTO) desde archivo Monitor METS-DIRECTOS
import pandas as pd
import os

# Ruta del archivo Monitor METS-DIRECTOS
archivo_monitor = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 25. Monitor METS\Monitor METS-DIRECTOS 2026.xlsx"

print(f"\n{'='*60}")
print(f"IDENTIFICANDO TIPO DE CLIENTE (MET/DIRECTO)")
print(f"{'='*60}")

if os.path.exists(archivo_monitor):
    print(f"✅ Archivo encontrado: {os.path.basename(archivo_monitor)}")
    
    # Leer la hoja "METS" - Columnas A (Nombre Ruta) y D (Tipo de Cliente)
    df_monitor_mets = pd.read_excel(
        archivo_monitor,
        sheet_name='METS',
        engine='openpyxl',
        usecols="A,D"  # Columnas A y D
    )
    
    print(f"📋 Columnas leídas de hoja METS: {list(df_monitor_mets.columns)}")
    print(f"📊 Total de registros en Monitor: {len(df_monitor_mets)}")
    
    # Renombrar columnas para claridad (ajustar según nombres reales en el archivo)
    # Asumiendo que columna A es el nombre de ruta y D es el tipo
    columna_ruta = df_monitor_mets.columns[0]  # Primera columna (A)
    columna_tipo = df_monitor_mets.columns[1]  # Segunda columna (D)
    
    df_monitor_mets = df_monitor_mets.rename(columns={
        columna_ruta: 'Nombre Ruta',
        columna_tipo: 'Tipo Cliente'
    })
    
    # Eliminar duplicados y valores nulos
    df_monitor_mets = df_monitor_mets.dropna(subset=['Nombre Ruta'])
    df_monitor_mets = df_monitor_mets.drop_duplicates(subset=['Nombre Ruta'])
    
    print(f"📌 Registros únicos después de limpieza: {len(df_monitor_mets)}")
    
    # Agregar columna "Tipo Cliente" al DataFrame principal
    if 'df_filtrado_sf' in locals() and not df_filtrado_sf.empty:
        
        # --- REGLA PARA CDG: BUSCARX desde Monitor METS ---
        # Hacer merge usando VLOOKUP equivalente para registros CDG
        df_filtrado_sf = df_filtrado_sf.merge(
            df_monitor_mets[['Nombre Ruta', 'Tipo Cliente']], 
            on='Nombre Ruta', 
            how='left'
        )
        
        # Rellenar valores nulos con "-" (equivalente a la función BUSCARX de Excel)
        df_filtrado_sf['Tipo Cliente'] = df_filtrado_sf['Tipo Cliente'].fillna('-')
        
        # --- REGLA PARA SFOR: Lógica condicional basada en Nombre Ruta ---
        # Excel: =SI(O(ESNUMERO(HALLAR("-",G2)),ESNUMERO(HALLAR("RECOGE",G2))),0,1)
        # Si contiene "-" o "RECOGE" → 0, si no → 1
        
        def clasificar_sfor(row):
            """
            Aplica regla de clasificación para SFOR:
            - Si 'Nombre Ruta' contiene "-" o "RECOGE" → retorna 0
            - Si no contiene ninguno → retorna 1
            """
            if row['Origen'] != 'SFOR':
                return row['Tipo Cliente']  # Mantener valor de CDG
            
            nombre_ruta = str(row['Nombre Ruta']).upper()
            
            # Verificar si contiene "-" o "RECOGE"
            if '-' in nombre_ruta or 'RECOGE' in nombre_ruta:
                return '0'
            else:
                return '1'
        
        # Aplicar la función a todas las filas
        df_filtrado_sf['Tipo Cliente'] = df_filtrado_sf.apply(clasificar_sfor, axis=1)
        
        print(f"\n✅ Columna 'Tipo Cliente' agregada correctamente")
        print(f"   📌 CDG: Obtenido desde Monitor METS (BUSCARX)")
        print(f"   📌 SFOR: Calculado por regla (contiene '-' o 'RECOGE' → 0, caso contrario → 1)")
        print(f"\n📊 Distribución de tipos de cliente:")
        print(df_filtrado_sf['Tipo Cliente'].value_counts())
        
        # Mostrar distribución por origen y tipo
        if 'Origen' in df_filtrado_sf.columns:
            print(f"\n📈 Distribución por Origen y Tipo Cliente:")
            print(df_filtrado_sf.groupby(['Origen', 'Tipo Cliente']).size())
    else:
        print("❌ El DataFrame df_filtrado_sf no existe o está vacío")
        
else:
    print(f"❌ Archivo no encontrado: {archivo_monitor}")
    print("⚠️  Continuando sin identificación de tipo de cliente...")
    # Agregar columna vacía para mantener consistencia
    if 'df_filtrado_sf' in locals():
        df_filtrado_sf['Tipo Cliente'] = '-'


IDENTIFICANDO TIPO DE CLIENTE (MET/DIRECTO)
✅ Archivo encontrado: Monitor METS-DIRECTOS 2026.xlsx
📋 Columnas leídas de hoja METS: ['Nombre de ruta CIAT', 'Tipo']
📊 Total de registros en Monitor: 351
📌 Registros únicos después de limpieza: 299

✅ Columna 'Tipo Cliente' agregada correctamente
   📌 CDG: Obtenido desde Monitor METS (BUSCARX)
   📌 SFOR: Calculado por regla (contiene '-' o 'RECOGE' → 0, caso contrario → 1)

📊 Distribución de tipos de cliente:
Tipo Cliente
0          509
1          248
-           86
DIRECTO     18
METS         7
Name: count, dtype: int64

📈 Distribución por Origen y Tipo Cliente:
Origen  Tipo Cliente
CDG     -                86
        DIRECTO          18
        METS              7
SFOR    0               509
        1               248
dtype: int64


3. Filtrar según tipo de cliente (CDG: MET/DIRECTO, SFOR: solo 1)

In [3]:
# Filtrar registros según tipo de cliente
# CDG: Solo MET o DIRECTO
# SFOR: Solo los que tienen valor 1

print(f"\n{'='*60}")
print(f"FILTRADO POR TIPO DE CLIENTE")
print(f"{'='*60}")

if 'df_filtrado_sf' in locals() and not df_filtrado_sf.empty and 'Tipo Cliente' in df_filtrado_sf.columns:
    
    filas_antes = len(df_filtrado_sf)
    
    # Mostrar valores únicos ANTES del filtro para diagnóstico
    print(f"\n🔍 DIAGNÓSTICO - Valores únicos de 'Tipo Cliente' por Origen:")
    print(df_filtrado_sf.groupby(['Origen', 'Tipo Cliente']).size())
    
    # Crear filtros según origen (usando .str.upper() para evitar problemas de mayúsculas/minúsculas)
    filtro_cdg = (df_filtrado_sf['Origen'] == 'CDG') & (
        df_filtrado_sf['Tipo Cliente'].astype(str).str.upper().isin(['METS', 'DIRECTO'])
    )
    filtro_sfor = (df_filtrado_sf['Origen'] == 'SFOR') & (df_filtrado_sf['Tipo Cliente'] == '1')
    
    # Combinar filtros con OR (mantener los que cumplan cualquiera de las dos condiciones)
    filtro_final = filtro_cdg | filtro_sfor
    
    # Guardar filas eliminadas para análisis
    df_eliminados_tipo = df_filtrado_sf[~filtro_final].copy()
    
    # Aplicar filtro
    df_filtrado_sf = df_filtrado_sf[filtro_final].copy().reset_index(drop=True)
    
    filas_despues = len(df_filtrado_sf)
    filas_eliminadas = filas_antes - filas_despues
    
    print(f"\n📊 Filas antes del filtro: {filas_antes}")
    print(f"✅ Filas después del filtro: {filas_despues}")
    print(f"🗑️  Filas eliminadas: {filas_eliminadas}")
    
    # Mostrar distribución final
    print(f"\n📈 Distribución final por Origen y Tipo Cliente (CONSERVADOS):")
    if not df_filtrado_sf.empty:
        print(df_filtrado_sf.groupby(['Origen', 'Tipo Cliente']).size())
    
    # Mostrar qué se eliminó
    if not df_eliminados_tipo.empty:
        print(f"\n🗑️  Tipos de cliente eliminados:")
        print(df_eliminados_tipo.groupby(['Origen', 'Tipo Cliente']).size())
        
else:
    print("❌ No se puede aplicar filtro: falta columna 'Tipo Cliente' o DataFrame vacío")


FILTRADO POR TIPO DE CLIENTE

🔍 DIAGNÓSTICO - Valores únicos de 'Tipo Cliente' por Origen:
Origen  Tipo Cliente
CDG     -                86
        DIRECTO          18
        METS              7
SFOR    0               509
        1               248
dtype: int64

📊 Filas antes del filtro: 868
✅ Filas después del filtro: 273
🗑️  Filas eliminadas: 595

📈 Distribución final por Origen y Tipo Cliente (CONSERVADOS):
Origen  Tipo Cliente
CDG     DIRECTO          18
        METS              7
SFOR    1               248
dtype: int64

🗑️  Tipos de cliente eliminados:
Origen  Tipo Cliente
CDG     -                86
SFOR    0               509
dtype: int64


4. Búsqueda MAESTRA en SARTT - Extraer SAP, Unidad, Tiros y Volumen (Una sola consulta)

In [4]:
# BÚSQUEDA MAESTRA EN SARTT - Una sola consulta para extraer SAP, Unidad, Tiros y Volumen
import time
import pandas as pd
from io import StringIO
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

print(f"\n{'='*80}")
print(f"BÚSQUEDA MAESTRA EN SARTT - OPTIMIZADA")
print(f"{'='*80}")
print(f"Estrategia: 1 sola consulta para extraer SAP, Unidad, Tiros y Volumen")
print(f"{'='*80}\n")

# Configuración
USUARIO = "IGCAMPOSG"
CONTRASENA = "Carcacha21"
URL_LOGIN = "http://sartt.truper.com/sartt/login.jspx"
URL_REPORTE = "http://sartt.truper.com/sartt/reportes/reporteRecuperacionUnidadesPrincipal.jspx"
LOTE_SIZE = 100

# Obtener todas las rutas únicas del DataFrame filtrado
if 'df_filtrado_sf' in locals() and not df_filtrado_sf.empty:
    todas_las_rutas = df_filtrado_sf['Ruta'].dropna().unique().tolist()
    
    print(f"📊 Total de rutas únicas a buscar: {len(todas_las_rutas)}")
    
    # Dividir en lotes de 100
    lotes = [todas_las_rutas[i:i + LOTE_SIZE] for i in range(0, len(todas_las_rutas), LOTE_SIZE)]
    print(f"📦 Número de lotes: {len(lotes)} (de {LOTE_SIZE} rutas c/u)")
    print(f"⚡ Consultas estimadas: {len(lotes)} (vs 10+ con el método anterior)")
    print(f"{'='*80}\n")
    
    # DataFrame para almacenar TODOS los resultados
    resultados_sartt_master = pd.DataFrame()
    
    # PROCESAR CADA LOTE
    for num_lote, lote_rutas in enumerate(lotes, 1):
        print(f"{'='*80}")
        print(f"LOTE {num_lote}/{len(lotes)} - Procesando {len(lote_rutas)} rutas")
        print(f"{'='*80}")
        
        driver = webdriver.Chrome()
        wait = WebDriverWait(driver, 30)
        
        try:
            rutas_texto = '\n'.join(map(str, lote_rutas))
            
            # Login
            print("🔐 Iniciando sesión...")
            driver.get(URL_LOGIN)
            time.sleep(2)
            
            # Esperar que desaparezca loading
            try:
                for _ in range(10):
                    loading_elements = driver.find_elements(By.ID, "loading")
                    if not loading_elements or not loading_elements[0].is_displayed():
                        break
                    time.sleep(1)
            except:
                pass
            
            # Manejar modal-backdrop si existe
            try:
                modal_backdrop = driver.find_elements(By.CLASS_NAME, "modal-backdrop")
                if modal_backdrop:
                    botones_cerrar = driver.find_elements(By.XPATH, "//button[contains(text(), 'Aceptar') or contains(text(), 'OK')]")
                    if botones_cerrar:
                        botones_cerrar[0].click()
                        time.sleep(1)
            except:
                pass
            
            wait.until(EC.element_to_be_clickable((By.ID, "loginForm:j_username"))).send_keys(USUARIO)
            driver.find_element(By.ID, "loginForm:j_password").send_keys(CONTRASENA)
            wait.until(EC.element_to_be_clickable((By.ID, "loginForm:j_id45"))).click()
            
            # Manejar modal de sesión duplicada
            time.sleep(2)
            try:
                modal_backdrop = driver.find_elements(By.CLASS_NAME, "modal-backdrop")
                if modal_backdrop:
                    botones_confirmacion = driver.find_elements(By.XPATH, "//button[contains(text(), 'Sí') or contains(text(), 'Aceptar')]")
                    if botones_confirmacion:
                        botones_confirmacion[0].click()
                        time.sleep(2)
            except:
                pass
            
            wait.until(EC.visibility_of_element_located((By.ID, "formaPrincipal:icePnlTbSet:0:dashboard")))
            print("✅ Sesión iniciada")
            
            # Navegar al reporte
            driver.get(URL_REPORTE)
            time.sleep(2)
            try:
                for _ in range(10):
                    loading_elements = driver.find_elements(By.ID, "loading")
                    if not loading_elements or not loading_elements[0].is_displayed():
                        break
                    time.sleep(1)
            except:
                pass
            
            # Seleccionar tipo CIAT
            select_element = wait.until(EC.presence_of_element_located((By.ID, "forma:icePnlTbSet:0:j_id200")))
            Select(select_element).select_by_value("2")
            time.sleep(0.5)
            
            # Ingresar rutas
            cuadro_rutas = wait.until(EC.element_to_be_clickable((By.ID, "forma:icePnlTbSet:0:j_id204")))
            cuadro_rutas.clear()
            time.sleep(0.2)
            cuadro_rutas.send_keys(rutas_texto)
            time.sleep(0.3)
            cuadro_rutas.send_keys(Keys.TAB)
            time.sleep(0.2)
            
            # Eliminar modal-backdrop si está bloqueando
            try:
                modal_backdrop = driver.find_elements(By.CLASS_NAME, "modal-backdrop")
                if modal_backdrop:
                    for modal in modal_backdrop:
                        try:
                            driver.execute_script("arguments[0].remove();", modal)
                        except:
                            pass
                    time.sleep(1)
            except:
                pass
            
            # Hacer clic en Buscar
            boton_buscar = None
            try:
                wait.until(lambda d: len(d.find_elements(By.CLASS_NAME, "modal-backdrop")) == 0)
            except:
                pass
            
            try:
                boton_buscar = driver.find_element(By.ID, "forma:icePnlTbSet:0:j_id204")
            except:
                try:
                    boton_buscar = driver.find_element(By.XPATH, "//input[@type='submit' and contains(@value, 'Buscar')]")
                except:
                    botones = driver.find_elements(By.XPATH, "//input[@type='submit']")
                    if botones:
                        boton_buscar = botones[0]
            
            if boton_buscar:
                try:
                    driver.execute_script("arguments[0].scrollIntoView(true);", boton_buscar)
                    time.sleep(0.5)
                    boton_buscar.click()
                    print(f"✅ Búsqueda iniciada")
                except:
                    try:
                        driver.execute_script("arguments[0].click();", boton_buscar)
                        print(f"✅ Búsqueda iniciada (JavaScript)")
                    except:
                        print(f"⚠️  HAZ CLIC MANUAL EN 'BUSCAR'")
                        time.sleep(5)
            
            # Esperar resultados
            print(f"⏳ Esperando resultados...")
            time.sleep(2)
            try:
                for _ in range(15):
                    loading_elements = driver.find_elements(By.ID, "loading")
                    if not loading_elements or not loading_elements[0].is_displayed():
                        break
                    time.sleep(1)
            except:
                pass
            time.sleep(2)
            
            # Extraer tabla
            id_tabla_resultados = "forma:icePnlTbSet:0:rutas"
            try:
                tabla_elemento = None
                try:
                    tabla_elemento = wait.until(EC.visibility_of_element_located((By.ID, id_tabla_resultados)))
                except:
                    time.sleep(3)
                    tabla_elemento = driver.find_element(By.ID, id_tabla_resultados)
                
                if tabla_elemento:
                    tabla_html = tabla_elemento.get_attribute('outerHTML')
                    
                    if not tabla_html or len(tabla_html) < 100:
                        time.sleep(3)
                        tabla_html = tabla_elemento.get_attribute('outerHTML')
                    
                    # Extraer datos
                    tablas_encontradas = pd.read_html(StringIO(tabla_html))
                    if tablas_encontradas:
                        df_lote = tablas_encontradas[0]
                        
                        if isinstance(df_lote.columns, pd.MultiIndex):
                            df_lote.columns = ['_'.join(col).strip() for col in df_lote.columns.values]
                        
                        df_lote.columns = df_lote.columns.str.replace(r'[^\w\s]', '', regex=True)
                        df_lote.columns = df_lote.columns.str.strip()
                        
                        # Consolidar resultados
                        if resultados_sartt_master.empty:
                            resultados_sartt_master = df_lote
                        else:
                            resultados_sartt_master = pd.concat([resultados_sartt_master, df_lote], ignore_index=True)
                        
                        print(f"✅ Extraídas {len(df_lote)} filas")
                        print(f"📊 Total acumulado: {len(resultados_sartt_master)} filas")
                    else:
                        print(f"⚠️  No se encontraron resultados")
                        
            except Exception as e:
                print(f"❌ Error al extraer tabla: {e}")
        
        except Exception as e:
            print(f"❌ Error en lote {num_lote}: {e}")
        
        finally:
            # Cerrar sesión y navegador
            try:
                driver.delete_all_cookies()
            except:
                pass
            try:
                driver.quit()
                print(f"🔒 Navegador cerrado")
            except:
                pass
            
            # Pausa entre lotes
            if num_lote < len(lotes):
                time.sleep(3)
            print()
    
    # RESUMEN FINAL
    print(f"\n{'='*80}")
    print(f"✅ BÚSQUEDA MAESTRA COMPLETADA")
    print(f"{'='*80}")
    print(f"📊 Rutas buscadas: {len(todas_las_rutas)}")
    print(f"📦 Lotes procesados: {len(lotes)}")
    print(f"✅ Filas extraídas: {len(resultados_sartt_master)}")
    
    if not resultados_sartt_master.empty:
        print(f"\n📋 Columnas disponibles en SARTT:")
        for i, col in enumerate(resultados_sartt_master.columns, 1):
            print(f"   {i:2d}. {col}")
        
        # Identificar columnas clave
        columna_ruta_sartt = None
        columna_sap_sartt = None
        columna_unidad_sartt = None
        columna_tiros_sartt = None
        columna_volumen_sartt = None
        
        for col in resultados_sartt_master.columns:
            col_upper = col.upper()
            if 'CIAT' in col_upper and 'RUTA' in col_upper:
                columna_ruta_sartt = col
            elif 'SAP' in col_upper and 'RUTA' in col_upper:
                columna_sap_sartt = col
            elif 'UNIDAD' in col_upper:
                columna_unidad_sartt = col
            elif 'TIRO' in col_upper:
                columna_tiros_sartt = col
            elif 'VOLUMEN' in col_upper and 'M3' in col_upper:
                columna_volumen_sartt = col
        
        print(f"\n🔍 Columnas identificadas:")
        print(f"   {'Ruta:':<15} {columna_ruta_sartt or '❌ No encontrada'}")
        print(f"   {'SAP:':<15} {columna_sap_sartt or '❌ No encontrada'}")
        print(f"   {'Unidad:':<15} {columna_unidad_sartt or '❌ No encontrada'}")
        print(f"   {'Tiros:':<15} {columna_tiros_sartt or '❌ No encontrada'}")
        print(f"   {'Volumen m3:':<15} {columna_volumen_sartt or '❌ No encontrada'}")
    
    print(f"{'='*80}")
    
else:
    print("❌ El DataFrame df_filtrado_sf no existe o está vacío")
    resultados_sartt_master = pd.DataFrame()



BÚSQUEDA MAESTRA EN SARTT - OPTIMIZADA
Estrategia: 1 sola consulta para extraer SAP, Unidad, Tiros y Volumen

📊 Total de rutas únicas a buscar: 273
📦 Número de lotes: 3 (de 100 rutas c/u)
⚡ Consultas estimadas: 3 (vs 10+ con el método anterior)

LOTE 1/3 - Procesando 100 rutas


There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


🔐 Iniciando sesión...
✅ Sesión iniciada
✅ Búsqueda iniciada
⏳ Esperando resultados...
✅ Extraídas 100 filas
📊 Total acumulado: 100 filas
🔒 Navegador cerrado

LOTE 2/3 - Procesando 100 rutas


There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


🔐 Iniciando sesión...
✅ Sesión iniciada
✅ Búsqueda iniciada
⏳ Esperando resultados...
✅ Extraídas 100 filas
📊 Total acumulado: 200 filas
🔒 Navegador cerrado

LOTE 3/3 - Procesando 73 rutas


There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


🔐 Iniciando sesión...
✅ Sesión iniciada
✅ Búsqueda iniciada
⏳ Esperando resultados...
✅ Extraídas 73 filas
📊 Total acumulado: 273 filas
🔒 Navegador cerrado


✅ BÚSQUEDA MAESTRA COMPLETADA
📊 Rutas buscadas: 273
📦 Lotes procesados: 3
✅ Filas extraídas: 273

📋 Columnas disponibles en SARTT:
    1. Rutas_SART_SART
    2. Rutas_CIAT_CIAT
    3. Rutas_SAP_SAP
    4. Rutas_Nombre_Nombre
    5. Rutas_Tipo_Tipo
    6. Rutas_Facturable_Peso kg
    7. Rutas_Facturable_Volumen m3
    8. Rutas_Entregable_Peso kg
    9. Rutas_Entregable_Volumen m3
   10. Rutas_Tiros_Tiros
   11. Rutas_Partidas_Partidas
   12. Rutas_Placa_Placa
   13. Rutas_Línea_Línea
   14. Rutas_Fecha Enrampe_Fecha Enrampe
   15. Rutas_Anden_Anden
   16. Rutas_Surtidores_Surtidores
   17. Rutas_Autorizador_Autorizador
   18. Rutas_Fecha Solicitud Transporte_Fecha Solicitud Transporte
   19. Rutas_Planeación_Planeación
   20. Rutas_En Patio_En Patio
   21. Rutas_Asignación_Asignación
   22. Rutas_Liberación_Liberación
   23. Rutas_

5. Completar SAP, Unidad, Tiros y Volumen usando resultados de SARTT

In [5]:
# COMPLETAR SAP, UNIDAD, TIROS Y VOLUMEN usando resultados de SARTT Master
import pandas as pd

print(f"\n{'='*80}")
print(f"COMPLETANDO INFORMACIÓN DESDE RESULTADOS SARTT MASTER")
print(f"{'='*80}\n")

if 'df_filtrado_sf' in locals() and 'resultados_sartt_master' in locals() and not resultados_sartt_master.empty:
    
    # Crear copia del DataFrame
    tiempos_limpio = df_filtrado_sf.copy()
    
    # Normalizar columnas Ruta (quitar ceros a la izquierda)
    print("📋 Normalizando columnas Ruta...")
    tiempos_limpio['Ruta'] = tiempos_limpio['Ruta'].astype(str).str.lstrip('0').replace('', '0')
    
    # Identificar columnas en resultados_sartt_master
    columna_ruta_sartt = None
    columna_sap_sartt = None
    columna_unidad_sartt = None
    columna_tiros_sartt = None
    columna_volumen_sartt = None
    
    for col in resultados_sartt_master.columns:
        col_upper = col.upper()
        if 'CIAT' in col_upper and 'RUTA' in col_upper:
            columna_ruta_sartt = col
        elif 'SAP' in col_upper and 'RUTA' in col_upper:
            columna_sap_sartt = col
        elif 'UNIDAD' in col_upper:
            columna_unidad_sartt = col
        elif 'TIRO' in col_upper:
            columna_tiros_sartt = col
        elif 'VOLUMEN' in col_upper and 'M3' in col_upper:
            columna_volumen_sartt = col
    
    if columna_ruta_sartt:
        # Normalizar rutas en SARTT
        resultados_sartt_norm = resultados_sartt_master.copy()
        resultados_sartt_norm[columna_ruta_sartt] = resultados_sartt_norm[columna_ruta_sartt].astype(str).str.lstrip('0')
        
        # Preparar DataFrame para merge
        columnas_merge = [columna_ruta_sartt]
        rename_dict = {columna_ruta_sartt: 'Ruta'}
        
        if columna_sap_sartt:
            columnas_merge.append(columna_sap_sartt)
            rename_dict[columna_sap_sartt] = 'SAP_SARTT'
        if columna_unidad_sartt:
            columnas_merge.append(columna_unidad_sartt)
            rename_dict[columna_unidad_sartt] = 'Unidad_SARTT'
        if columna_tiros_sartt:
            columnas_merge.append(columna_tiros_sartt)
            rename_dict[columna_tiros_sartt] = 'Tiros'
        if columna_volumen_sartt:
            columnas_merge.append(columna_volumen_sartt)
            rename_dict[columna_volumen_sartt] = 'Volumen m3'
        
        df_sartt_merge = resultados_sartt_norm[columnas_merge].copy()
        df_sartt_merge = df_sartt_merge.drop_duplicates(subset=[columna_ruta_sartt])
        df_sartt_merge = df_sartt_merge.rename(columns=rename_dict)
        
        # Hacer merge con tiempos_limpio
        tiempos_limpio = tiempos_limpio.merge(df_sartt_merge, on='Ruta', how='left')
        
        # ===== COMPLETAR SAP =====
        if 'SAP_SARTT' in tiempos_limpio.columns:
            print(f"\n{'='*60}")
            print("COMPLETANDO SAP")
            print(f"{'='*60}")
            
            # Identificar SAP vacíos ANTES
            sap_vacios_antes = (
                tiempos_limpio['SAP'].isna() |
                (tiempos_limpio['SAP'].astype(str).str.lower().isin(['nan'])) |
                (tiempos_limpio['SAP'].astype(str).str.strip() == '') |
                (tiempos_limpio['SAP'].astype(str).str.strip() == '0')
            ).sum()
            
            print(f"❌ SAP vacíos ANTES: {sap_vacios_antes}")
            
            # Completar SAP donde esté vacío
            mascara_sap_vacio = (
                tiempos_limpio['SAP'].isna() |
                (tiempos_limpio['SAP'].astype(str).str.lower().isin(['nan'])) |
                (tiempos_limpio['SAP'].astype(str).str.strip() == '') |
                (tiempos_limpio['SAP'].astype(str).str.strip() == '0')
            )
            
            tiempos_limpio.loc[mascara_sap_vacio, 'SAP'] = tiempos_limpio.loc[mascara_sap_vacio, 'SAP_SARTT']
            
            # Identificar SAP vacíos DESPUÉS
            sap_vacios_despues = (
                tiempos_limpio['SAP'].isna() |
                (tiempos_limpio['SAP'].astype(str).str.lower().isin(['nan'])) |
                (tiempos_limpio['SAP'].astype(str).str.strip() == '') |
                (tiempos_limpio['SAP'].astype(str).str.strip() == '0')
            ).sum()
            
            filas_completadas_sap = sap_vacios_antes - sap_vacios_despues
            print(f"✅ SAP completados: {filas_completadas_sap}")
            print(f"❌ SAP vacíos DESPUÉS: {sap_vacios_despues}")
            print(f"📊 Tasa de éxito: {filas_completadas_sap/sap_vacios_antes*100:.1f}%" if sap_vacios_antes > 0 else "")
            
            # Eliminar columna temporal
            tiempos_limpio = tiempos_limpio.drop(columns=['SAP_SARTT'])
        
        # ===== AGREGAR UNIDAD DESDE CATÁLOGO =====
        print(f"\n{'='*60}")
        print("AGREGANDO UNIDAD DESDE CATÁLOGO")
        print(f"{'='*60}")
        
        archivo_catalogo = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - Catalogos de Placas\Consolidado catalogo de placas.xlsx"
        
        import os
        if os.path.exists(archivo_catalogo):
            df_catalogo = pd.read_excel(archivo_catalogo, sheet_name='Catalogo consolidado', engine='openpyxl')
            
            if 'Placa' in df_catalogo.columns and 'Unidad' in df_catalogo.columns:
                df_catalogo = df_catalogo.drop_duplicates(subset=['Placa'])
                
                # Identificar columna LT (Línea 2.0)
                columna_lt = None
                for col in df_catalogo.columns:
                    if 'Línea' in str(col) or 'Linea' in str(col) or col == df_catalogo.columns[4]:
                        columna_lt = col
                        break
                
                if columna_lt is None and len(df_catalogo.columns) > 4:
                    columna_lt = df_catalogo.columns[4]
                
                columnas_merge_cat = ['Placa', 'Unidad']
                if columna_lt is not None:
                    df_catalogo = df_catalogo.rename(columns={columna_lt: 'LT'})
                    columnas_merge_cat.append('LT')
                
                tiempos_limpio = tiempos_limpio.merge(df_catalogo[columnas_merge_cat], on='Placa', how='left')
                
                if 'Unidad' in tiempos_limpio.columns:
                    tiempos_limpio['Unidad'] = tiempos_limpio['Unidad'].fillna('Sin unidad')
                    
                    # Reemplazar valores específicos
                    tiempos_limpio['Unidad'] = tiempos_limpio['Unidad'].replace({
                        '53 ft': 'C53',
                        '40 ft H.C': 'C40',
                        '48 ft': 'C48'
                    })
                    
                    unidades_encontradas = (tiempos_limpio['Unidad'] != 'Sin unidad').sum()
                    print(f"✅ Unidades desde catálogo: {unidades_encontradas}/{len(tiempos_limpio)}")
                
                if 'LT' in tiempos_limpio.columns:
                    tiempos_limpio['LT'] = tiempos_limpio['LT'].fillna('Otras líneas')
                    print(f"✅ Columna LT agregada")
        
        # ===== COMPLETAR UNIDAD FALTANTE DESDE SARTT =====
        if 'Unidad_SARTT' in tiempos_limpio.columns and 'Unidad' in tiempos_limpio.columns:
            print(f"\n{'='*60}")
            print("COMPLETANDO UNIDAD DESDE SARTT")
            print(f"{'='*60}")
            
            sin_unidad_antes = (tiempos_limpio['Unidad'] == 'Sin unidad').sum()
            print(f"❌ Sin unidad ANTES: {sin_unidad_antes}")
            
            # Completar donde Unidad = 'Sin unidad'
            mascara_sin_unidad = tiempos_limpio['Unidad'] == 'Sin unidad'
            tiempos_limpio.loc[mascara_sin_unidad, 'Unidad'] = tiempos_limpio.loc[mascara_sin_unidad, 'Unidad_SARTT']
            
            # Volver a marcar como 'Sin unidad' si SARTT tampoco tiene dato
            tiempos_limpio.loc[tiempos_limpio['Unidad'].isna(), 'Unidad'] = 'Sin unidad'
            
            sin_unidad_despues = (tiempos_limpio['Unidad'] == 'Sin unidad').sum()
            unidades_completadas = sin_unidad_antes - sin_unidad_despues
            print(f"✅ Unidades completadas desde SARTT: {unidades_completadas}")
            print(f"❌ Sin unidad DESPUÉS: {sin_unidad_despues}")
            
            # Eliminar columna temporal
            tiempos_limpio = tiempos_limpio.drop(columns=['Unidad_SARTT'])
        
        # ===== VERIFICAR TIROS Y VOLUMEN =====
        if 'Tiros' in tiempos_limpio.columns:
            print(f"\n{'='*60}")
            print("TIROS Y VOLUMEN")
            print(f"{'='*60}")
            tiros_encontrados = tiempos_limpio['Tiros'].notna().sum()
            print(f"✅ Tiros encontrados: {tiros_encontrados}/{len(tiempos_limpio)}")
        
        if 'Volumen m3' in tiempos_limpio.columns:
            volumen_encontrado = tiempos_limpio['Volumen m3'].notna().sum()
            print(f"✅ Volumen encontrado: {volumen_encontrado}/{len(tiempos_limpio)}")
        
        # RESUMEN FINAL
        print(f"\n{'='*80}")
        print("✅ COMPLETADO DE INFORMACIÓN FINALIZADO")
        print(f"{'='*80}")
        print(f"📊 Total de filas: {len(tiempos_limpio)}")
        print(f"📋 Total de columnas: {len(tiempos_limpio.columns)}")
        print(f"\n📋 Columnas finales:")
        for i, col in enumerate(tiempos_limpio.columns, 1):
            print(f"   {i:2d}. {col}")
        print(f"{'='*80}")
        
    else:
        print("❌ No se encontró la columna de Ruta en resultados_sartt_master")
        tiempos_limpio = df_filtrado_sf.copy()
        
else:
    print("❌ No se puede completar información:")
    if 'df_filtrado_sf' not in locals():
        print("   - df_filtrado_sf no existe")
    if 'resultados_sartt_master' not in locals():
        print("   - resultados_sartt_master no existe")
    elif resultados_sartt_master.empty:
        print("   - resultados_sartt_master está vacío")
    
    if 'df_filtrado_sf' in locals():
        tiempos_limpio = df_filtrado_sf.copy()


COMPLETANDO INFORMACIÓN DESDE RESULTADOS SARTT MASTER

📋 Normalizando columnas Ruta...

COMPLETANDO SAP
❌ SAP vacíos ANTES: 214
✅ SAP completados: 214
❌ SAP vacíos DESPUÉS: 0
📊 Tasa de éxito: 100.0%

AGREGANDO UNIDAD DESDE CATÁLOGO
✅ Unidades desde catálogo: 87/273
✅ Columna LT agregada

COMPLETANDO UNIDAD DESDE SARTT
❌ Sin unidad ANTES: 186
✅ Unidades completadas desde SARTT: 186
❌ Sin unidad DESPUÉS: 0

TIROS Y VOLUMEN
✅ Tiros encontrados: 273/273
✅ Volumen encontrado: 273/273

✅ COMPLETADO DE INFORMACIÓN FINALIZADO
📊 Total de filas: 273
📋 Total de columnas: 14

📋 Columnas finales:
    1. Tipo de Unidad
    2. Ruta
    3. Ruta Conca
    4. Nombre Ruta
    5. SAP
    6. Placa
    7. Cierre
    8. Escaneos
    9. Origen
   10. Tipo Cliente
   11. Tiros
   12. Volumen m3
   13. Unidad
   14. LT


6. Agregar columna Cliente y Nombre cliente desde Monitor METS

In [6]:
# Agregar columnas Cliente y Nombre cliente desde Monitor METS (VLOOKUP)
import pandas as pd
import os

archivo_monitor_mets = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 25. Monitor METS\Monitor METS-DIRECTOS 2026.xlsx"

print("="*80)
print("AGREGANDO COLUMNAS CLIENTE Y NOMBRE CLIENTE DESDE MONITOR METS")
print("="*80)

if os.path.exists(archivo_monitor_mets):
    print(f"✅ Archivo encontrado: {os.path.basename(archivo_monitor_mets)}")
    
    # Leer la hoja 'METS' - Columnas A, B y C
    df_mets = pd.read_excel(
        archivo_monitor_mets,
        sheet_name='METS',
        engine='openpyxl'
    )
    
    # Mostrar información del DataFrame METS
    print(f"\n📋 Información del archivo METS:")
    print(f"   Filas: {len(df_mets)}")
    print(f"   Columnas totales: {len(df_mets.columns)}")
    
    # Verificar que las columnas A, B y C existan (índices 0, 1 y 2)
    if len(df_mets.columns) >= 3:
        columna_a = df_mets.columns[0]  # Primera columna (A) - Nombre Ruta
        columna_b = df_mets.columns[1]  # Segunda columna (B) - Cliente
        columna_c = df_mets.columns[2]  # Tercera columna (C) - Nombre cliente
        
        print(f"\n   ✅ Columna A (índice 0): '{columna_a}'")
        print(f"   ✅ Columna B (índice 1): '{columna_b}'")
        print(f"   ✅ Columna C (índice 2): '{columna_c}'")
        
        # Mostrar primeras filas para diagnóstico
        print(f"\n📊 Primeras 5 filas de las columnas A, B y C:")
        print(df_mets[[columna_a, columna_b, columna_c]].head())
        
        # Verificar que tiempos_limpio existe
        if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
            
            # Eliminar columnas si ya existen (para evitar duplicados)
            columnas_eliminar = [col for col in ['Cliente', 'Nombre cliente'] if col in tiempos_limpio.columns]
            if columnas_eliminar:
                print(f"\n⚠️  Columnas {columnas_eliminar} ya existen, eliminándolas para evitar duplicados...")
                tiempos_limpio = tiempos_limpio.drop(columns=columnas_eliminar)
            
            # Verificar que existe columna 'Nombre Ruta' en tiempos_limpio
            if 'Nombre Ruta' in tiempos_limpio.columns:
                print(f"\n🔍 Columna 'Nombre Ruta' encontrada en tiempos_limpio")
                print(f"   Valores únicos en 'Nombre Ruta': {tiempos_limpio['Nombre Ruta'].nunique()}")
                
                # Preparar DataFrame METS para el merge (columnas A, B y C)
                df_mets_merge = df_mets[[columna_a, columna_b, columna_c]].copy()
                df_mets_merge = df_mets_merge.drop_duplicates(subset=[columna_a])
                
                # Renombrar columnas para facilitar el merge
                df_mets_merge = df_mets_merge.rename(columns={
                    columna_a: 'Nombre Ruta',
                    columna_b: 'Cliente',
                    columna_c: 'Nombre cliente'
                })
                
                # Eliminar filas con valores nulos en 'Nombre Ruta'
                df_mets_merge = df_mets_merge.dropna(subset=['Nombre Ruta'])
                
                print(f"\n📊 DataFrame METS preparado para merge:")
                print(f"   Filas después de eliminar duplicados y nulos: {len(df_mets_merge)}")
                print(f"   Valores únicos de 'Nombre Ruta': {df_mets_merge['Nombre Ruta'].nunique()}")
                
                # Hacer el merge tipo VLOOKUP (left join)
                filas_antes = len(tiempos_limpio)
                tiempos_limpio = tiempos_limpio.merge(
                    df_mets_merge,
                    on='Nombre Ruta',
                    how='left'
                )
                filas_despues = len(tiempos_limpio)
                
                # Verificar resultados del merge
                if 'Cliente' in tiempos_limpio.columns and 'Nombre cliente' in tiempos_limpio.columns:
                    clientes_encontrados = tiempos_limpio['Cliente'].notna().sum()
                    nombres_encontrados = tiempos_limpio['Nombre cliente'].notna().sum()
                    
                    print(f"\n✅ Columnas agregadas exitosamente")
                    print(f"   📊 Filas antes del merge: {filas_antes}")
                    print(f"   📊 Filas después del merge: {filas_despues}")
                    print(f"   ✅ Clientes encontrados: {clientes_encontrados}/{len(tiempos_limpio)} ({clientes_encontrados/len(tiempos_limpio)*100:.1f}%)")
                    print(f"   ✅ Nombres cliente encontrados: {nombres_encontrados}/{len(tiempos_limpio)} ({nombres_encontrados/len(tiempos_limpio)*100:.1f}%)")
                    
                    # Mostrar algunos ejemplos
                    print(f"\n📋 Ejemplos de merge (primeras 5 filas con Cliente):")
                    ejemplos = tiempos_limpio[tiempos_limpio['Cliente'].notna()][['Nombre Ruta', 'Cliente', 'Nombre cliente']].head()
                    if not ejemplos.empty:
                        print(ejemplos.to_string(index=False))
                    
                    # Verificar columnas finales
                    print(f"\n📋 VERIFICACIÓN POST-MERGE:")
                    print(f"   Total de columnas en tiempos_limpio: {len(tiempos_limpio.columns)}")
                    
                else:
                    print(f"\n❌ Error: Las columnas 'Cliente' y/o 'Nombre cliente' no se crearon después del merge")
                    
            else:
                print(f"\n❌ Error: No se encontró la columna 'Nombre Ruta' en tiempos_limpio")
                print(f"   Columnas disponibles: {list(tiempos_limpio.columns)}")
                
        else:
            print(f"\n❌ Error: El DataFrame tiempos_limpio no existe o está vacío")
            
    else:
        print(f"\n❌ Error: El archivo METS no tiene al menos 3 columnas")
        print(f"   Columnas encontradas: {len(df_mets.columns)}")
        
else:
    print(f"❌ Archivo no encontrado: {archivo_monitor_mets}")

print("="*80)

AGREGANDO COLUMNAS CLIENTE Y NOMBRE CLIENTE DESDE MONITOR METS
✅ Archivo encontrado: Monitor METS-DIRECTOS 2026.xlsx

📋 Información del archivo METS:
   Filas: 351
   Columnas totales: 9

   ✅ Columna A (índice 0): 'Nombre de ruta CIAT'
   ✅ Columna B (índice 1): 'Clave cliente'
   ✅ Columna C (índice 2): 'Nombre cliente'

📊 Primeras 5 filas de las columnas A, B y C:
                 Nombre de ruta CIAT  Clave cliente  \
0              VILLAHERMOSA VAQUEIRO         588816   
1            VALENCIA MIGUEL GABRIEL         581649   
2          TIENDAS MIKASA, S.A. DE C         538760   
3  TIENDAS FIX,   S. DE R.L. DE C.V.         559600   
4         Sanver Forte, S.A. de C.V.         552462   

                      Nombre cliente  
0              Villahermosa Vaqueiro  
1            VALENCIA MIGUEL GABRIEL  
2          TIENDAS MIKASA, S.A. DE C  
3  TIENDAS FIX,   S. DE R.L. DE C.V.  
4                       Sanver Forte  

🔍 Columna 'Nombre Ruta' encontrada en tiempos_limpio
   Valores 

7. Corregir columna Tipo Cliente usando datos de METS

In [7]:
# Corregir columna Tipo Cliente usando columna D de METS
import pandas as pd
import os

archivo_monitor_mets = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 25. Monitor METS\Monitor METS-DIRECTOS 2026.xlsx"

print("="*80)
print("CORRIGIENDO COLUMNA TIPO CLIENTE USANDO DATOS DE METS")
print("="*80)

if os.path.exists(archivo_monitor_mets):
    print(f"✅ Archivo encontrado: {os.path.basename(archivo_monitor_mets)}")
    
    # Leer la hoja 'METS'
    df_mets = pd.read_excel(
        archivo_monitor_mets,
        sheet_name='METS',
        engine='openpyxl'
    )
    
    # Verificar que las columnas A y D existan (índices 0 y 3)
    if len(df_mets.columns) >= 4:
        columna_a = df_mets.columns[0]  # Primera columna (A) - Nombre de ruta
        columna_d = df_mets.columns[3]  # Cuarta columna (D) - Tipo Cliente
        
        print(f"\n📋 Columnas identificadas en METS:")
        print(f"   ✅ Columna A (índice 0): '{columna_a}'")
        print(f"   ✅ Columna D (índice 3): '{columna_d}'")
        
        # Mostrar primeras filas para diagnóstico
        print(f"\n📊 Primeras 5 filas de las columnas A y D:")
        print(df_mets[[columna_a, columna_d]].head())
        
        # Verificar que tiempos_limpio existe
        if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
            
            # Verificar que existe columna 'Tipo Cliente' en tiempos_limpio
            if 'Tipo Cliente' in tiempos_limpio.columns:
                print(f"\n🔍 Columna 'Tipo Cliente' encontrada en tiempos_limpio")
                
                # Contar valores actuales de Tipo Cliente
                valores_tipo_cliente = tiempos_limpio['Tipo Cliente'].value_counts()
                print(f"\n📊 Distribución actual de 'Tipo Cliente':")
                print(valores_tipo_cliente)
                
                # Contar cuántos tienen valor 1 (convertir a string para comparación)
                mascara_valor_1 = tiempos_limpio['Tipo Cliente'].astype(str) == '1'
                filas_con_valor_1 = mascara_valor_1.sum()
                print(f"\n⚠️  Filas con 'Tipo Cliente' = 1: {filas_con_valor_1}")
                
                if filas_con_valor_1 > 0:
                    # Preparar DataFrame METS para el mapeo
                    df_mets_d = df_mets[[columna_a, columna_d]].copy()
                    df_mets_d = df_mets_d.drop_duplicates(subset=[columna_a])
                    
                    # Renombrar columnas
                    df_mets_d = df_mets_d.rename(columns={
                        columna_a: 'Nombre Ruta',
                        columna_d: 'Tipo Cliente Nuevo'
                    })
                    
                    # Eliminar filas con valores nulos
                    df_mets_d = df_mets_d.dropna(subset=['Nombre Ruta'])
                    
                    print(f"\n📊 DataFrame METS preparado:")
                    print(f"   Filas después de eliminar duplicados y nulos: {len(df_mets_d)}")
                    print(f"   Valores únicos de 'Tipo Cliente Nuevo': {df_mets_d['Tipo Cliente Nuevo'].nunique()}")
                    
                    # Crear diccionario de mapeo
                    mapeo_tipo_cliente = dict(zip(df_mets_d['Nombre Ruta'], df_mets_d['Tipo Cliente Nuevo']))
                    
                    print(f"\n🔄 Aplicando correcciones...")
                    
                    # Contador de cambios
                    filas_actualizadas = 0
                    
                    # Iterar sobre las filas con Tipo Cliente = 1 (usando máscara)
                    for idx, row in tiempos_limpio[mascara_valor_1].iterrows():
                        nombre_ruta = row['Nombre Ruta']
                        
                        if nombre_ruta in mapeo_tipo_cliente:
                            nuevo_tipo = mapeo_tipo_cliente[nombre_ruta]
                            
                            # Verificar que el nuevo tipo no sea nulo ni vacío
                            if pd.notna(nuevo_tipo) and str(nuevo_tipo).strip() != '':
                                tiempos_limpio.at[idx, 'Tipo Cliente'] = nuevo_tipo
                                filas_actualizadas += 1
                    
                    # Mostrar resultados
                    print(f"\n✅ Actualización completada:")
                    print(f"   Filas actualizadas: {filas_actualizadas}/{filas_con_valor_1}")
                    
                    # Mostrar nueva distribución
                    valores_tipo_cliente_final = tiempos_limpio['Tipo Cliente'].value_counts()
                    print(f"\n📊 Distribución final de 'Tipo Cliente':")
                    print(valores_tipo_cliente_final)
                    
                    # Verificar si aún quedan valores = 1 (convertir a string)
                    filas_con_valor_1_final = (tiempos_limpio['Tipo Cliente'].astype(str) == '1').sum()
                    if filas_con_valor_1_final > 0:
                        print(f"\n⚠️  Filas que aún tienen 'Tipo Cliente' = 1: {filas_con_valor_1_final}")
                        print(f"   (No se encontró coincidencia en METS para estas rutas)")
                        
                        # Mostrar algunas rutas sin coincidencia
                        rutas_sin_coincidencia = tiempos_limpio[tiempos_limpio['Tipo Cliente'].astype(str) == '1']['Nombre Ruta'].unique()[:5]
                        print(f"   Ejemplos de rutas sin coincidencia:")
                        for ruta in rutas_sin_coincidencia:
                            print(f"      - {ruta}")
                    else:
                        print(f"\n🎉 Todas las filas con valor 1 fueron actualizadas exitosamente")
                    
                    # Mostrar ejemplos de cambios
                    print(f"\n📋 Ejemplos de correcciones:")
                    ejemplos = tiempos_limpio[tiempos_limpio['Nombre Ruta'].isin(list(mapeo_tipo_cliente.keys())[:5])][['Nombre Ruta', 'Tipo Cliente']].head()
                    if not ejemplos.empty:
                        print(ejemplos)
                        
                else:
                    print(f"\n✅ No hay filas con 'Tipo Cliente' = 1, no se requieren correcciones")
                    
            else:
                print(f"\n❌ Error: No se encontró la columna 'Tipo Cliente' en tiempos_limpio")
                print(f"   Columnas disponibles: {list(tiempos_limpio.columns)}")
                
        else:
            print(f"\n❌ Error: El DataFrame tiempos_limpio no existe o está vacío")
            
    else:
        print(f"\n❌ Error: El archivo METS no tiene al menos 4 columnas")
        print(f"   Columnas encontradas: {len(df_mets.columns)}")
        
else:
    print(f"❌ Archivo no encontrado: {archivo_monitor_mets}")

print("="*80)

CORRIGIENDO COLUMNA TIPO CLIENTE USANDO DATOS DE METS
✅ Archivo encontrado: Monitor METS-DIRECTOS 2026.xlsx

📋 Columnas identificadas en METS:
   ✅ Columna A (índice 0): 'Nombre de ruta CIAT'
   ✅ Columna D (índice 3): 'Tipo'

📊 Primeras 5 filas de las columnas A y D:
                 Nombre de ruta CIAT     Tipo
0              VILLAHERMOSA VAQUEIRO  DIRECTO
1            VALENCIA MIGUEL GABRIEL  DIRECTO
2          TIENDAS MIKASA, S.A. DE C  DIRECTO
3  TIENDAS FIX,   S. DE R.L. DE C.V.  DIRECTO
4         Sanver Forte, S.A. de C.V.  DIRECTO

🔍 Columna 'Tipo Cliente' encontrada en tiempos_limpio

📊 Distribución actual de 'Tipo Cliente':
Tipo Cliente
1          248
DIRECTO     18
METS         7
Name: count, dtype: int64

⚠️  Filas con 'Tipo Cliente' = 1: 248

📊 DataFrame METS preparado:
   Filas después de eliminar duplicados y nulos: 299
   Valores únicos de 'Tipo Cliente Nuevo': 2

🔄 Aplicando correcciones...

✅ Actualización completada:
   Filas actualizadas: 236/248

📊 Distribución fin

8. Eliminar filas donde Tiros ≠ 1

In [8]:
# Eliminar filas donde Tiros es diferente de 1
import pandas as pd

print("="*80)
print("ELIMINANDO FILAS DONDE TIROS ≠ 1")
print("="*80)

if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
    
    # Buscar la columna Tiros (puede ser 'Tiros', 'Tiros_x', 'Tiros_y', etc.)
    columna_tiros = None
    for col in tiempos_limpio.columns:
        if col.startswith('Tiros'):
            columna_tiros = col
            print(f"✅ Columna encontrada: '{columna_tiros}'")
            break
    
    if columna_tiros:
        
        # Información antes de la eliminación
        filas_antes = len(tiempos_limpio)
        print(f"\n📊 Estado ANTES de la eliminación:")
        print(f"   Total de filas: {filas_antes}")
        
        # Mostrar distribución de valores en Tiros
        print(f"\n📋 Distribución de valores en '{columna_tiros}':")
        distribucion_tiros = tiempos_limpio[columna_tiros].value_counts().sort_index()
        print(distribucion_tiros)
        
        # Contar valores nulos
        nulos = tiempos_limpio[columna_tiros].isna().sum()
        if nulos > 0:
            print(f"\n⚠️  Valores nulos en '{columna_tiros}': {nulos}")
        
        # Contar filas con Tiros = 1 (convertir a numérico para comparación)
        try:
            # Intentar convertir a numérico
            tiros_numericos = pd.to_numeric(tiempos_limpio[columna_tiros], errors='coerce')
            filas_con_1 = (tiros_numericos == 1).sum()
            filas_a_eliminar = len(tiempos_limpio) - filas_con_1
            
            print(f"\n🔍 Análisis:")
            print(f"   ✅ Filas con Tiros = 1 (se conservarán): {filas_con_1}")
            print(f"   ❌ Filas con Tiros ≠ 1 (se eliminarán): {filas_a_eliminar}")
            
            # Mostrar ejemplos de filas a eliminar
            if filas_a_eliminar > 0:
                print(f"\n📋 Ejemplos de filas que se eliminarán (primeras 5):")
                filas_eliminar = tiempos_limpio[tiros_numericos != 1][['Ruta', 'Nombre Ruta', columna_tiros]].head()
                if not filas_eliminar.empty:
                    print(filas_eliminar)
            
            # Filtrar para mantener solo Tiros = 1
            tiempos_limpio = tiempos_limpio[tiros_numericos == 1].copy()
            
            # Información después de la eliminación
            filas_despues = len(tiempos_limpio)
            filas_eliminadas = filas_antes - filas_despues
            
            print(f"\n{'='*80}")
            print(f"RESUMEN DE ELIMINACIÓN")
            print(f"{'='*80}")
            print(f"📊 Filas ANTES: {filas_antes}")
            print(f"📊 Filas DESPUÉS: {filas_despues}")
            print(f"❌ Filas eliminadas: {filas_eliminadas}")
            
            if filas_antes > 0:
                print(f"✅ Porcentaje conservado: {filas_despues/filas_antes*100:.1f}%")
            
            # Verificar distribución final
            if not tiempos_limpio.empty:
                print(f"\n📋 Distribución final de '{columna_tiros}':")
                distribucion_final = tiempos_limpio[columna_tiros].value_counts().sort_index()
                print(distribucion_final)
            
            print(f"\n✅ Filtrado completado exitosamente")
            
        except Exception as e:
            print(f"\n❌ Error al procesar la columna '{columna_tiros}': {e}")
            print(f"   Tipo de error: {type(e).__name__}")
            
    else:
        print(f"\n❌ Error: No se encontró ninguna columna que comience con 'Tiros' en tiempos_limpio")
        print(f"   Columnas disponibles: {list(tiempos_limpio.columns)}")
        
else:
    print(f"\n❌ Error: El DataFrame tiempos_limpio no existe o está vacío")

print(f"{'='*80}")

ELIMINANDO FILAS DONDE TIROS ≠ 1
✅ Columna encontrada: 'Tiros'

📊 Estado ANTES de la eliminación:
   Total de filas: 273

📋 Distribución de valores en 'Tiros':
Tiros
1     260
5       1
6       1
7       5
8       2
9       1
10      3
Name: count, dtype: int64

🔍 Análisis:
   ✅ Filas con Tiros = 1 (se conservarán): 260
   ❌ Filas con Tiros ≠ 1 (se eliminarán): 13

📋 Ejemplos de filas que se eliminarán (primeras 5):
     Ruta                    Nombre Ruta  Tiros
28  60311      COLIN ESQUIVEL MARTIN JUS      7
52  61141           ROBERTO LUGO SANCHEZ      6
85  60180        CRISTIAN ZARZA ESQUIVEL      7
92  59643  MIGUEL ANGEL GONZALEZ ANGELES      8
94  59708            ARSEN, S.A. DE C.V.     10

RESUMEN DE ELIMINACIÓN
📊 Filas ANTES: 273
📊 Filas DESPUÉS: 260
❌ Filas eliminadas: 13
✅ Porcentaje conservado: 95.2%

📋 Distribución final de 'Tiros':
Tiros
1    260
Name: count, dtype: int64

✅ Filtrado completado exitosamente


8.1. Agregar columnas de fecha derivadas de Cierre

In [9]:
# Agregar columnas de fecha derivadas de la columna Cierre
import pandas as pd
import locale

print("="*80)
print("AGREGANDO COLUMNAS DE FECHA DERIVADAS DE CIERRE")
print("="*80)

if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
    
    # Verificar que existe la columna Cierre
    if 'Cierre' in tiempos_limpio.columns:
        print(f"✅ Columna 'Cierre' encontrada")
        
        # Configurar locale a español para nombres de meses
        try:
            locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
        except:
            try:
                locale.setlocale(locale.LC_TIME, 'Spanish_Spain.1252')
            except:
                print("⚠️  No se pudo configurar locale español, usando nombres manuales")
        
        # Convertir columna Cierre a datetime si no lo es
        if not pd.api.types.is_datetime64_any_dtype(tiempos_limpio['Cierre']):
            print(f"\n🔄 Convirtiendo columna 'Cierre' a datetime...")
            tiempos_limpio['Cierre'] = pd.to_datetime(tiempos_limpio['Cierre'], errors='coerce')
            
            # Verificar conversión
            nulos = tiempos_limpio['Cierre'].isna().sum()
            if nulos > 0:
                print(f"⚠️  {nulos} valores no se pudieron convertir a fecha")
        
        # Diccionario de meses en español (por si locale no funciona)
        meses_es = {
            1: 'ene', 2: 'feb', 3: 'mar', 4: 'abr', 5: 'may', 6: 'jun',
            7: 'jul', 8: 'ago', 9: 'sep', 10: 'oct', 11: 'nov', 12: 'dic'
        }
        
        print(f"\n📅 Creando columnas derivadas...")
        
        # 1. Fecha Cierre: formato dd-mmm (ej: 15-dic)
        tiempos_limpio['Fecha Cierre'] = tiempos_limpio['Cierre'].apply(
            lambda x: f"{x.day:02d}-{meses_es[x.month]}" if pd.notna(x) else None
        )
        print(f"   ✅ Fecha Cierre (dd-mmm)")
        
        # 2. Semana: formato S + número de semana (ej: S50)
        tiempos_limpio['Semana'] = tiempos_limpio['Cierre'].apply(
            lambda x: f"S{x.isocalendar()[1]}" if pd.notna(x) else None
        )
        print(f"   ✅ Semana (S + número)")
        
        # 3. Mes: formato mmm-aa (ej: dic-25)
        tiempos_limpio['Mes'] = tiempos_limpio['Cierre'].apply(
            lambda x: f"{meses_es[x.month]}-{str(x.year)[-2:]}" if pd.notna(x) else None
        )
        print(f"   ✅ Mes (mmm-aa)")
        
        # 4. Mes2: formato mmm (ej: dic)
        tiempos_limpio['Mes2'] = tiempos_limpio['Cierre'].apply(
            lambda x: meses_es[x.month] if pd.notna(x) else None
        )
        print(f"   ✅ Mes2 (mmm)")
        
        # 5. Año: formato aaaa (ej: 2025)
        tiempos_limpio['Año'] = tiempos_limpio['Cierre'].apply(
            lambda x: x.year if pd.notna(x) else None
        )
        print(f"   ✅ Año (aaaa)")
        
        # Mostrar ejemplos
        print(f"\n📋 Ejemplos de columnas creadas (primeras 5 filas):")
        columnas_fecha = ['Cierre', 'Fecha Cierre', 'Semana', 'Mes', 'Mes2', 'Año']
        columnas_mostrar = [col for col in columnas_fecha if col in tiempos_limpio.columns]
        
        if columnas_mostrar:
            ejemplos = tiempos_limpio[columnas_mostrar].head()
            print(ejemplos.to_string(index=False))
        
        # Resumen
        print(f"\n{'='*80}")
        print("RESUMEN")
        print(f"{'='*80}")
        print(f"✅ Total de columnas agregadas: 5")
        print(f"📊 Total de columnas en tiempos_limpio: {len(tiempos_limpio.columns)}")
        print(f"📋 Nuevas columnas:")
        print(f"   1. Fecha Cierre (dd-mmm)")
        print(f"   2. Semana (S + número)")
        print(f"   3. Mes (mmm-aa)")
        print(f"   4. Mes2 (mmm)")
        print(f"   5. Año (aaaa)")
        
    else:
        print(f"❌ Error: No se encontró la columna 'Cierre' en tiempos_limpio")
        print(f"   Columnas disponibles: {list(tiempos_limpio.columns)}")
        
else:
    print(f"❌ Error: El DataFrame tiempos_limpio no existe o está vacío")

print(f"{'='*80}")

AGREGANDO COLUMNAS DE FECHA DERIVADAS DE CIERRE
✅ Columna 'Cierre' encontrada

📅 Creando columnas derivadas...
   ✅ Fecha Cierre (dd-mmm)
   ✅ Semana (S + número)
   ✅ Mes (mmm-aa)
   ✅ Mes2 (mmm)
   ✅ Año (aaaa)

📋 Ejemplos de columnas creadas (primeras 5 filas):
             Cierre Fecha Cierre Semana    Mes Mes2  Año
2026-03-16 09:00:50       16-mar    S12 mar-26  mar 2026
2026-03-13 03:59:11       13-mar    S11 mar-26  mar 2026
2026-03-11 01:33:04       11-mar    S11 mar-26  mar 2026
2026-03-12 03:23:00       12-mar    S11 mar-26  mar 2026
2026-03-13 04:21:50       13-mar    S11 mar-26  mar 2026

RESUMEN
✅ Total de columnas agregadas: 5
📊 Total de columnas en tiempos_limpio: 21
📋 Nuevas columnas:
   1. Fecha Cierre (dd-mmm)
   2. Semana (S + número)
   3. Mes (mmm-aa)
   4. Mes2 (mmm)
   5. Año (aaaa)


8.2. Limpiar y separar columna Placa

In [10]:
# Limpiar y separar columna Placa (si tiene formato "XX / YY", separar en dos columnas)
import pandas as pd

print("="*80)
print("LIMPIANDO Y SEPARANDO COLUMNA PLACA")
print("="*80)

if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
    
    # Verificar que existe la columna Placa
    if 'Placa' in tiempos_limpio.columns:
        print(f"✅ Columna 'Placa' encontrada")
        
        # Crear columna 'Placa 2' inicialmente vacía
        tiempos_limpio['Placa 2'] = None
        
        # Contador de placas separadas
        placas_separadas = 0
        
        # Mostrar algunos ejemplos ANTES de la limpieza
        print(f"\n📋 Ejemplos de placas ANTES de la limpieza (primeras 10):")
        placas_con_barra = tiempos_limpio[tiempos_limpio['Placa'].astype(str).str.contains('/', na=False)]['Placa'].head(10)
        if not placas_con_barra.empty:
            for i, placa in enumerate(placas_con_barra, 1):
                print(f"   {i:2d}. {placa}")
        else:
            print("   No se encontraron placas con '/'")
        
        print(f"\n🔄 Procesando columna Placa...")
        
        # Iterar sobre todas las filas
        for idx, row in tiempos_limpio.iterrows():
            placa_valor = str(row['Placa'])
            
            # Verificar si la placa contiene "/"
            if '/' in placa_valor:
                # Separar por "/"
                partes = placa_valor.split('/')
                
                if len(partes) >= 2:
                    # Primera parte se queda en Placa (limpiar espacios)
                    tiempos_limpio.at[idx, 'Placa'] = partes[0].strip()
                    
                    # Segunda parte va a Placa 2 (limpiar espacios)
                    tiempos_limpio.at[idx, 'Placa 2'] = partes[1].strip()
                    
                    placas_separadas += 1
        
        # Estadísticas
        total_placas = len(tiempos_limpio)
        placas_con_valor_2 = tiempos_limpio['Placa 2'].notna().sum()
        
        print(f"\n{'='*80}")
        print("RESUMEN DE LIMPIEZA DE PLACAS")
        print(f"{'='*80}")
        print(f"📊 Total de filas: {total_placas}")
        print(f"✅ Placas separadas: {placas_separadas}")
        print(f"📋 Placas con valor en 'Placa 2': {placas_con_valor_2}")
        
        # Mostrar algunos ejemplos DESPUÉS de la limpieza
        if placas_con_valor_2 > 0:
            print(f"\n📋 Ejemplos DESPUÉS de la limpieza (primeras 10):")
            ejemplos = tiempos_limpio[tiempos_limpio['Placa 2'].notna()][['Placa', 'Placa 2']].head(10)
            print(ejemplos.to_string(index=False))
        
        # Verificar columnas finales
        print(f"\n📋 Total de columnas en tiempos_limpio: {len(tiempos_limpio.columns)}")
        print(f"✅ Columna 'Placa 2' agregada exitosamente")
        
    else:
        print(f"❌ Error: No se encontró la columna 'Placa' en tiempos_limpio")
        print(f"   Columnas disponibles: {list(tiempos_limpio.columns)}")
        
else:
    print(f"❌ Error: El DataFrame tiempos_limpio no existe o está vacío")

print(f"{'='*80}")

LIMPIANDO Y SEPARANDO COLUMNA PLACA
✅ Columna 'Placa' encontrada

📋 Ejemplos de placas ANTES de la limpieza (primeras 10):
    1. 11UU4V / 54BB5P
    2. 18UR7Y / 05AZ5Z
    3. 19US6V / 17AU3Z
    4. 19US6V / 17AU3Z
    5. 19US6V / 17AU3Z
    6. 19US6V / 17AU3Z
    7. 26TX3V / 39BJ5U
    8. 36UX9W / 17AU3Z
    9. 36UX9W / 17AU3Z
   10. 41UX3B / 37BC6J

🔄 Procesando columna Placa...

RESUMEN DE LIMPIEZA DE PLACAS
📊 Total de filas: 260
✅ Placas separadas: 19
📋 Placas con valor en 'Placa 2': 19

📋 Ejemplos DESPUÉS de la limpieza (primeras 10):
 Placa Placa 2
11UU4V  54BB5P
18UR7Y  05AZ5Z
19US6V  17AU3Z
19US6V  17AU3Z
19US6V  17AU3Z
19US6V  17AU3Z
26TX3V  39BJ5U
36UX9W  17AU3Z
36UX9W  17AU3Z
41UX3B  37BC6J

📋 Total de columnas en tiempos_limpio: 22
✅ Columna 'Placa 2' agregada exitosamente


9. Imprimir CSV

9.1. Validación - Exportar todos los DataFrames para validar secuencia

In [11]:
# Exportar todos los DataFrames creados hasta este momento para validación
import pandas as pd
import os
from datetime import datetime

# Ruta de salida
ruta_outputs = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta METSyDIRECTOS\Ouputs"
os.makedirs(ruta_outputs, exist_ok=True)

# Generar nombre con fecha y hora
fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
nombre_archivo = f"Validacion_Secuencia_Completa_{fecha_actual}.xlsx"
ruta_completa = os.path.join(ruta_outputs, nombre_archivo)

print(f"\n{'='*80}")
print(f"EXPORTANDO TODOS LOS DATAFRAMES PARA VALIDACIÓN DE SECUENCIA")
print(f"{'='*80}")

# Crear un diccionario con todos los DataFrames disponibles en orden cronológico
dataframes_validacion = {}

# 1. DataFrame inicial consolidado (CDG + SFOR)
if 'df_sep' in locals() and not df_sep.empty:
    dataframes_validacion['01_Consolidado_CDG_SFOR'] = df_sep
    print(f"✅ 01. df_sep (Consolidado inicial): {len(df_sep)} filas")

# 2. DataFrame después del filtrado por tipo de cliente
if 'df_filtrado_sf' in locals() and not df_filtrado_sf.empty:
    dataframes_validacion['02_Filtrado_TipoCliente'] = df_filtrado_sf
    print(f"✅ 02. df_filtrado_sf (Post-filtro): {len(df_filtrado_sf)} filas")

# 3. Monitor METS (referencia para tipo cliente)
if 'df_monitor_mets' in locals() and not df_monitor_mets.empty:
    dataframes_validacion['03_Monitor_METS'] = df_monitor_mets
    print(f"✅ 03. df_monitor_mets (Referencia): {len(df_monitor_mets)} filas")

# 4. Resultados SARTT Master (búsqueda única para SAP, Unidad, Tiros, Volumen)
if 'resultados_sartt_master' in locals() and not resultados_sartt_master.empty:
    dataframes_validacion['04_SARTT_Master'] = resultados_sartt_master
    print(f"✅ 04. resultados_sartt_master (SARTT Master): {len(resultados_sartt_master)} filas")

# 5. Catálogo de placas (referencia para Unidad y LT)
if 'df_catalogo' in locals() and not df_catalogo.empty:
    dataframes_validacion['05_Catalogo_Placas'] = df_catalogo
    print(f"✅ 05. df_catalogo (Catálogo): {len(df_catalogo)} filas")

# 6. DataFrame final con todas las columnas
if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
    dataframes_validacion['06_FINAL_Completo'] = tiempos_limpio
    print(f"✅ 06. tiempos_limpio (FINAL): {len(tiempos_limpio)} filas, {len(tiempos_limpio.columns)} columnas")
    print(f"    📋 Columnas: {list(tiempos_limpio.columns)}")

# Exportar a Excel con múltiples hojas
if dataframes_validacion:
    try:
        with pd.ExcelWriter(ruta_completa, engine='openpyxl') as writer:
            for nombre_hoja, df in dataframes_validacion.items():
                # Limitar nombre de hoja a 31 caracteres (límite de Excel)
                nombre_hoja_corto = nombre_hoja[:31]
                df.to_excel(writer, sheet_name=nombre_hoja_corto, index=False)
        
        print(f"\n{'='*80}")
        print(f"✅ ARCHIVO DE VALIDACIÓN EXPORTADO EXITOSAMENTE")
        print(f"{'='*80}")
        print(f"📁 Ruta: {ruta_completa}")
        print(f"📋 Total de hojas: {len(dataframes_validacion)}")
        print(f"🕒 Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"\n📊 RESUMEN DE SECUENCIA:")
        for i, (nombre, df) in enumerate(dataframes_validacion.items(), 1):
            print(f"   {i:2d}. {nombre[:40]:40s} - {len(df):6d} filas")
        print(f"{'='*80}")
        
    except Exception as e:
        print(f"❌ Error al exportar archivo: {e}")
else:
    print("⚠️  No hay DataFrames disponibles para exportar")


EXPORTANDO TODOS LOS DATAFRAMES PARA VALIDACIÓN DE SECUENCIA
✅ 01. df_sep (Consolidado inicial): 868 filas
✅ 02. df_filtrado_sf (Post-filtro): 273 filas
✅ 03. df_monitor_mets (Referencia): 299 filas
✅ 04. resultados_sartt_master (SARTT Master): 273 filas
✅ 05. df_catalogo (Catálogo): 2625 filas
✅ 06. tiempos_limpio (FINAL): 260 filas, 22 columnas
    📋 Columnas: ['Tipo de Unidad', 'Ruta', 'Ruta Conca', 'Nombre Ruta', 'SAP', 'Placa', 'Cierre', 'Escaneos', 'Origen', 'Tipo Cliente', 'Tiros', 'Volumen m3', 'Unidad', 'LT', 'Cliente', 'Nombre cliente', 'Fecha Cierre', 'Semana', 'Mes', 'Mes2', 'Año', 'Placa 2']

✅ ARCHIVO DE VALIDACIÓN EXPORTADO EXITOSAMENTE
📁 Ruta: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta METSyDIRECTOS\Ouputs\Validacion_Secuencia_Completa_20260317_231704.xlsx
📋 Total de hojas: 6
🕒 Fecha: 2026-03-17 23:17:06

📊 RESUMEN DE SECUENCIA:
    1. 01_Consolidado_CDG_SFOR                  -    868 filas
    2. 02_Filtrado_TipoCliente 

9.2. Imprimir CSV Final

In [12]:
#Imprimir y guardar tiempos_limpio
import os
from datetime import datetime

# Verificar que tenemos el DataFrame tiempos_limpio
if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
    
    # Si existe la columna 'Unidad', renombrarla a 'Unidad catalogo' para el CSV final
    if 'Unidad' in tiempos_limpio.columns:
        tiempos_limpio = tiempos_limpio.rename(columns={'Unidad': 'Unidad catalogo'})

    # Ruta de la carpeta de salida
    ruta_outputs = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta METSyDIRECTOS\Ouputs"
    
    # Crear la carpeta si no existe
    os.makedirs(ruta_outputs, exist_ok=True)
    
    # Generar nombre base con fecha y hora
    fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre_base = f"METSyDIRECTOS_Limpio_{fecha_actual}"
    extension = ".csv"
    
    # Función para encontrar el siguiente número consecutivo
    def obtener_nombre_unico(ruta_carpeta, nombre_base, extension):
        nombre_archivo = f"{nombre_base}{extension}"
        ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)
        
        # Si el archivo no existe, usar el nombre original
        if not os.path.exists(ruta_completa):
            return nombre_archivo, ruta_completa
        
        # Si existe, buscar el siguiente número consecutivo
        contador = 1
        while True:
            nombre_archivo = f"{nombre_base}({contador}){extension}"
            ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)
            
            if not os.path.exists(ruta_completa):
                return nombre_archivo, ruta_completa
            
            contador += 1
    
    # Obtener el nombre único
    nombre_archivo, ruta_completa = obtener_nombre_unico(ruta_outputs, nombre_base, extension)

    # Mostrar columnas disponibles ANTES de reordenar
    print(f"\n{'='*80}")
    print(f"COLUMNAS DISPONIBLES EN tiempos_limpio:")
    print(f"{'='*80}")
    print(f"Total de columnas: {len(tiempos_limpio.columns)}")
    for i, col in enumerate(tiempos_limpio.columns, 1):
        print(f"   {i:2d}. {col}")
    print(f"{'='*80}\n")

    # Definir orden exacto de columnas que se guardarán (SOLO estas columnas)
    columnas_orden = [
        'Tipo de Unidad',
        'Ruta',
        'Ruta Conca',
        'Nombre Ruta',
        'SAP',
        'Placa',
        'Cierre',
        'Escaneos',
        'Cliente',              # Cliente desde Monitor METS columna B
        'Nombre cliente',       # Nombre Cliente desde Monitor METS columna C
        'Tipo Cliente',         # Tipo Cliente
        'Origen',               # Nueva columna para identificar CDG/SFOR
        'Unidad catalogo',      # Unidad del catálogo
        'LT',                   # Línea 2.0 del catálogo
        'Volumen m3',           # Columna de SARTT
        'Fecha Cierre',         # Formato dd-mmm
        'Semana',               # Formato S + número de semana
        'Mes',                  # Formato mmm-aa
        'Mes2',                 # Formato mmm
        'Año',                  # Formato aaaa
        'Placa 2',              # Segunda placa (si hay "/" en Placa)
    ]
    
    # SOLO seleccionar las columnas que existen en el orden especificado
    # No incluir columnas adicionales como Tiros, Tipo Ruta, etc.
    columnas_finales = [col for col in columnas_orden if col in tiempos_limpio.columns]
    
    tiempos_limpio = tiempos_limpio[columnas_finales]
    
    print(f"📋 Orden final de columnas ({len(columnas_finales)}):")
    for i, col in enumerate(columnas_finales, 1):
        print(f"   {i:2d}. {col}")
    print(f"\n{'='*80}")

    # Guardar el DataFrame en CSV
    try:
        tiempos_limpio.to_csv(ruta_completa, index=False, encoding='utf-8-sig')
        print(f"✅ Archivo guardado exitosamente:")
        print(f"   📁 Carpeta: {ruta_outputs}")
        print(f"   📄 Archivo: {nombre_archivo}")
        print(f"   📊 Filas: {len(tiempos_limpio)}")
        print(f"   🕒 Fecha de guardado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
    except Exception as e:
        print(f"❌ Error al guardar el archivo: {e}")
        
else:
    print("❌ No se encontró el DataFrame tiempos_limpio o está vacío")
    if 'tiempos_limpio' not in locals():
        print("   El DataFrame tiempos_limpio no ha sido creado")
    elif tiempos_limpio.empty:
        print("   El DataFrame tiempos_limpio está vacío")


COLUMNAS DISPONIBLES EN tiempos_limpio:
Total de columnas: 22
    1. Tipo de Unidad
    2. Ruta
    3. Ruta Conca
    4. Nombre Ruta
    5. SAP
    6. Placa
    7. Cierre
    8. Escaneos
    9. Origen
   10. Tipo Cliente
   11. Tiros
   12. Volumen m3
   13. Unidad catalogo
   14. LT
   15. Cliente
   16. Nombre cliente
   17. Fecha Cierre
   18. Semana
   19. Mes
   20. Mes2
   21. Año
   22. Placa 2

📋 Orden final de columnas (21):
    1. Tipo de Unidad
    2. Ruta
    3. Ruta Conca
    4. Nombre Ruta
    5. SAP
    6. Placa
    7. Cierre
    8. Escaneos
    9. Cliente
   10. Nombre cliente
   11. Tipo Cliente
   12. Origen
   13. Unidad catalogo
   14. LT
   15. Volumen m3
   16. Fecha Cierre
   17. Semana
   18. Mes
   19. Mes2
   20. Año
   21. Placa 2

✅ Archivo guardado exitosamente:
   📁 Carpeta: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta METSyDIRECTOS\Ouputs
   📄 Archivo: METSyDIRECTOS_Limpio_20260317_231706.csv
   📊 Filas: 260
